# Suite SES — Candidates: a filter fed by a frame stream, ranked by a policy

"The transform candidates must come from a filter fed by a frame stream. That's how Store works, no?"
`Transforms.Session(transforms, frames)` is that filter ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)): `transforms`, visitors of
`Transforms.Transform`s, read once, when it is made; `frames`, the frame stream the caller supplies, each frame a
`Visitors.OfPattern`. The filter is each transform's prototype: a frame passes where it is a frame of the prototype.
A passing frame becomes candidates, one per combination of the values of the parameters whose domains are finite, any
other left open, in a canonical order; a follow-up call, `session.ranked(policy)`, ranks them by the policy, whose
clauses name transforms by object reference.
Frames are consumed once, and nothing is queried again: "In a streaming system termination conditions are
relative." The caller takes a candidate by applying it (`Transforms.apply`, then `Transforms.certify`) with its own visitors.

The tests play the caller: each makes its frame stream within its own transactions, from frames it obtained from its
stores (`support.bear`). The fixture is suite XF's shelf, with a note on each item, `shelf.Case` (a union of two
options, `upper` and `lower`) and `shelf.Size` (a union one of whose branches holds a value). Its patterns:
`shelf.Unlabelled`, an item without a label, declaring `case`; `shelf.Unnoted`, an item without a note, declaring
`loud` (a bool) and `note` (a str); `shelf.Unsized`, declaring `size` and `like` (for an object); `shelf.Any`, any
item; and `shelf.Whole`, the shelf.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Proxies, References
from mbse.Transforms import Transforms
from support import bear, raises


def shelf(txn):
    """shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name, a label and a note;
    shelf.Case, a union of two options; and shelf.Size, a union one of whose branches holds an item."""
    S = txn.Schemas
    return txn.uid("ses.shelf/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shelf": S.OfObject, "Item": S.OfObject, "Holds": S.OfRelation, "Upper": S.OfObject,
                  "Lower": S.OfObject, "Case": S.OfUnion, "Size": S.OfUnion})
        .spec([
            lambda _: _.Shelf.name("shelf.Shelf").ref(True).singleton("shelf.TheShelf").adjacencies({
                "items": lambda t: t.relation([_.Holds]).me("shelf")}),
            lambda _: _.Item.name("shelf.Item").ref(True).properties({
                "name": S.OfStr, "label": S.OfStr, "note": S.OfStr}).adjacencies([
                lambda t: t.name("shelf").relation([_.Holds]).me("item").owner(True)]),
            lambda _: _.Holds.name("shelf.Holds").links([lambda t: t.name("shelf"), lambda t: t.name("item")])
            .properties({"key": S.OfStr}).uniques([["shelf", "key"], ["item"]]),
            lambda _: _.Upper.name("shelf.Upper"),
            lambda _: _.Lower.name("shelf.Lower"),
            lambda _: _.Case.name("shelf.Case").branches([
                lambda b: b.name("upper").schema([_.Upper]), lambda b: b.name("lower").schema([_.Lower])]),
            lambda _: _.Size.name("shelf.Size").branches([
                lambda b: b.name("small").schema([_.Lower]), lambda b: b.name("custom").schema([_.Item])]),
        ])]))])


def patterns(txn):
    """shelf.Unlabelled, declaring case; shelf.Unnoted, declaring loud and note; shelf.Unsized, declaring size and
    like; shelf.Any; and shelf.Whole."""
    S, P = txn.Schemas, txn.Parametrics
    return txn.uid("ses.patterns/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Unlabelled": S.OfPattern, "Unnoted": S.OfPattern, "Unsized": S.OfPattern, "Any": S.OfPattern,
                  "Whole": S.OfPattern})
        .spec([
            lambda _: _.Unlabelled.name("shelf.Unlabelled").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.label(None)])
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "case": lambda d: d.parameter(lambda p: p.for_value([txn.shelf.Case]))}))]),
            lambda _: _.Unnoted.name("shelf.Unnoted").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.note(None)])
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "loud": lambda d: d.parameter(lambda p: p.for_value([S.OfBool])),
                "note": lambda d: d.parameter(lambda p: p.for_value([S.OfStr]))}))]),
            lambda _: _.Unsized.name("shelf.Unsized").symbols({"i": txn.shelf.Item})
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "size": lambda d: d.parameter(lambda p: p.for_value([txn.shelf.Size])),
                "like": lambda d: d.parameter(lambda p: p.for_object([txn.shelf.Item]))}))]),
            lambda _: _.Any.name("shelf.Any").symbols({"i": txn.shelf.Item}),
            lambda _: _.Whole.name("shelf.Whole").symbols({"s": txn.shelf.Shelf}),
        ])]))])


def model(*names):
    """A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it."""
    store = B.Store()
    store.transaction(Parametrics.install)
    store.transaction(shelf)
    store.transaction(patterns)
    store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"s": t.shelf.Shelf})]))]))
    for name in names:
        with store.transaction():
            store.shelf.Shelf(store.singleton("shelf.TheShelf"), lambda s, n=name: s.items({
                n: lambda e: e.item(lambda i: i.name(n))}))
    return store


def items(store):
    """Each item on the shelf: its key, name, label and note."""
    with store.transaction():
        return [(e.key.get(), e.item.name.get(), e.item.label.get(), e.item.note.get())
                for e in store.singleton("shelf.TheShelf").items]


def obtained(store, *names):
    """The frames of each pattern named, as `store.transaction` gives them back, each with its pattern's name."""
    return [(name, frame) for name in names for frame in store.transaction(
        lambda t, held=store.registered(name): t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))]


def stream(store, frames):
    """The caller's frame stream, within the store's transaction: each frame obtained, as a `Visitors.OfPattern`,
    made as it is pulled."""
    for name, frame in frames:
        yield bear(store, name, frame)


def counting(frames, pulled):
    """The frames obtained, each pattern's name appended to `pulled` as it is pulled."""
    for name, frame in frames:
        pulled.append(name)
        yield name, frame


def shown(groups):
    """Each frame's candidates: each one's transform's name, its frame's pattern and item, its assignments and its
    open parameters."""
    return [[(c.name, c.frame.as_schema().name, str(c.frame.i.name), dict(c.assignments), c.open) for c in group]
            for group in groups]


def catalog(store):
    """The catalog's transforms, by name, within the store's transaction."""
    return {str(e.name): e.transform for e in store.singleton("Transforms.TheCatalog").transforms}


transforms = model()  # the transforms' store: the schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install)
for define in (
    lambda t: Transforms.define(t, "Stock", "stock", t.shelf.Any),
    lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled),
    lambda t: Transforms.define(t, "Note", "note", t.shelf.Unnoted),
    lambda t: Transforms.define(t, "Size", "note", t.shelf.Unsized),
    lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled),
):
    transforms.transaction(define)
certs = B.Store()  # the certificates' store, as the caller gives it
certs.transaction(Transforms.install)
procedures = Transforms.Procedures()


def label(source, sync, target, assignments):
    """Labels the sync's item with its name, in the case its parameter assignment gives."""
    name = str(sync.i.name)
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.label(name.upper() if assignments["case"] == "upper" else name))


procedures.register("label", label)


def note(source, sync, target, assignments):
    """Notes the sync's item, loudly or not."""
    text = assignments["note"].upper() if assignments["loud"] else assignments["note"]
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.note(text))


procedures.register("note", note)


def copy(source, sync, target, assignments):
    """Puts on the target's shelf an item named as the source's, in the case its parameter assignment gives."""
    name = str(source.i.name)
    name = name.upper() if assignments["case"] == "upper" else name
    Proxies.store_of(target.s).shelf.Shelf(target.s, lambda s: s.items({name: lambda e: e.item(lambda i: i.name(name))}))


procedures.register("copy", copy)

<a id="SES-01"></a>
## SES-01 · The filter is each transform's prototype: a frame passes where it is a frame of the prototype, and becomes candidates, one per combination of its finite parameters' values (a bool's; a union of options' names), any other parameter left open, in the canonical order

In [ ]:
data = model("bolt", "nut")
frames = obtained(data, "shelf.Unlabelled", "shelf.Whole", "shelf.Unnoted", "shelf.Unsized", "shelf.Any")
with transforms.transaction(), data.transaction():
    named = catalog(transforms)
    session = Transforms.Session([named[n] for n in ("Stock", "Label", "Note", "Size")], stream(data, frames))
    groups = list(session)
    view = shown(groups)  # read within the caller's transaction, where its visitors live
assert view == [  # a frame's candidates: the transforms' order, then each value's in its domain
    [("Label", "shelf.Unlabelled", "bolt", {"case": "upper"}, ()), ("Label", "shelf.Unlabelled", "bolt", {"case": "lower"}, ())],
    [("Label", "shelf.Unlabelled", "nut", {"case": "upper"}, ()), ("Label", "shelf.Unlabelled", "nut", {"case": "lower"}, ())],
    # shelf.Whole's frame passes no filter
    [("Note", "shelf.Unnoted", "bolt", {"loud": False}, ("note",)), ("Note", "shelf.Unnoted", "bolt", {"loud": True}, ("note",))],
    [("Note", "shelf.Unnoted", "nut", {"loud": False}, ("note",)), ("Note", "shelf.Unnoted", "nut", {"loud": True}, ("note",))],
    [("Size", "shelf.Unsized", "bolt", {}, ("size", "like"))], [("Size", "shelf.Unsized", "nut", {}, ("size", "like"))],  # neither finite
    [("Stock", "shelf.Any", "bolt", {}, ())], [("Stock", "shelf.Any", "nut", {}, ())],  # no parameters
]
first = groups[0][0]
assert first.score == 0.0 and repr(first).startswith("Candidate(transform=") and "frame" not in repr(first)
loud = groups[2][1]
answered = loud.answer(note="fragile")  # an open parameter assigned
assert (dict(answered.assignments), answered.open) == ({"loud": True, "note": "fragile"}, ())
assert loud.open == ("note",)  # a candidate is a value: answering gives another
with raises(TypeError, match="transform 'Note' has no open parameter 'loud'"):
    loud.answer(loud=False)

<a id="SES-02"></a>
## SES-02 · A session consumes its stream: each frame is pulled once, as the caller iterates, and nothing is queried again; it ends when the stream does

In [ ]:
data = model("bolt", "nut")
frames = obtained(data, "shelf.Unlabelled")
pulled = []
with transforms.transaction(), data.transaction():
    session = Transforms.Session([catalog(transforms)["Label"]], stream(data, counting(frames, pulled)))
    assert pulled == []  # nothing pulled until the caller iterates
    each = iter(session)
    assert [str(c.frame.i.name) for c in next(each)] == ["bolt", "bolt"] and len(pulled) == 1  # one frame at a time
    assert [str(c.frame.i.name) for c in next(each)] == ["nut", "nut"] and len(pulled) == 2
    assert list(each) == [] and list(session) == []  # the stream drained: termination, relative to it
    assert list(session.ranked(Transforms.Policy())) == []  # iterated once, directly or ranked
assert len(pulled) == 2 and items(data) == [("bolt", "bolt", None, None), ("nut", "nut", None, None)]

<a id="SES-03"></a>
## SES-03 · A policy ranks each frame's candidates: a clause weighs the candidates of the transforms it names by object reference whose assignments hold its own; a score is the sum of the weights, ties in the canonical order

In [ ]:
data = model("bolt")
frames = obtained(data, "shelf.Unlabelled", "shelf.Unnoted")
with transforms.transaction(), data.transaction():
    named = catalog(transforms)
    Label, Copy, Note = (References.of(transforms, named[n]) for n in ("Label", "Copy", "Note"))
    policy = Transforms.Policy(
        Transforms.Clause(Label, {"case": "lower"}, 2.0),
        Transforms.Clause([Label, Copy], weight=0.5),  # one clause, several transforms: those sharing a reason
        Transforms.Clause(Note, {"note": "fragile"}),  # note is open: no candidate's assignments hold it
        Transforms.Clause(Copy, {"size": "small"}),  # Copy has no size: it weighs none of its candidates
    )
    ranked = list(Transforms.Session([named[n] for n in ("Copy", "Label", "Note")], stream(data, frames)).ranked(policy))
assert [[(c.name, dict(c.assignments), c.score) for c in group] for group in ranked] == [
    [("Label", {"case": "lower"}, 2.5),  # ranked first; then ties, in the canonical order
     ("Copy", {"case": "upper"}, 0.5), ("Copy", {"case": "lower"}, 0.5), ("Label", {"case": "upper"}, 0.5)],
    [("Note", {"loud": False}, 0.0), ("Note", {"loud": True}, 0.0)],
]
elsewhere = model()  # a reference is the catalog's: another store's transform named Label is another transform
elsewhere.transaction(Transforms.install)
elsewhere.transaction(lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled))
with elsewhere.transaction():
    other = References.of(elsewhere, catalog(elsewhere)["Label"])
assert Transforms.Policy(Transforms.Clause(other)).score(ranked[0][0]) == 0
with raises(TypeError, match="a clause names a transform by its reference, References.of(store, transform), or a sequence of them"):
    Transforms.Clause("Label")

<a id="SES-04"></a>
## SES-04 · The caller takes a candidate by applying it with its own visitors: the candidate's frame as the argument the caller chooses, its open parameters answered, and who decided it recorded

In [ ]:
source, sync, target = model("bolt", "nut"), model("bolt", "nut"), model()  # the caller's copies
in_sync, in_source = obtained(sync, "shelf.Unnoted"), obtained(source, "shelf.Unlabelled")
(_, shelf_frame), = obtained(target, "shelf.Whole")
with transforms.transaction(), sync.transaction():
    named = catalog(transforms)
    policy = Transforms.Policy(Transforms.Clause(References.of(transforms, named["Note"]), {"loud": True}))
    for group in Transforms.Session([named["Note"]], stream(sync, in_sync)).ranked(policy):
        taken = group[0].answer(note="fragile")  # the policy's first, its open parameter answered by the caller
        Transforms.apply(named["Note"], taken.assignments, procedures, sync=taken.frame)
        Transforms.certify(certs, named["Note"], taken.assignments, "taken", "policy", sync=taken.frame)
assert items(sync) == [("bolt", "bolt", None, "FRAGILE"), ("nut", "nut", None, "FRAGILE")] and items(source)[0][3] is None
with transforms.transaction(), source.transaction(), target.transaction():
    named = catalog(transforms)
    whole = bear(target, "shelf.Whole", shelf_frame)
    for group in Transforms.Session([named["Copy"]], stream(source, in_source)):
        if str(group[0].frame.i.name) == "nut":  # the caller decides: nut, upper; bolt left untaken
            Transforms.apply(named["Copy"], group[0].assignments, procedures, source=group[0].frame, target=whole)
            Transforms.certify(certs, named["Copy"], group[0].assignments, "taken", "caller", source=group[0].frame,
                               target=whole)
assert items(target) == [("NUT", "NUT", None, None)]
with certs.transaction():
    history = certs.singleton("Transforms.TheHistories").histories.at("taken").history
    assert [str(d.by) for d in history.decisions] == ["policy", "policy", "caller"]
    assert [[str(e.argument) for e in s.step.frames] for s in history.steps] == [["sync"], ["sync"], ["source", "target"]]